# Long-term dark-cycle experiment

Automated cycle for a cooled Toupcam on a Raspberry Pi. It images several stage positions in
complete darkness, at maximum gain, at the coldest sensor temperature, with the longest exposure.
For every position:

| # | Step | Bus power | Light |
|---|------|-----------|-------|
| 1 | Move to the preset X/Y (Z is **never** moved) | on | off |
| 2 | **Brightfield reference**: LED on, auto exposure, snap, LED off | on | LED |
| 3 | **Dark baseline**: bus off, wait 60 s, 30 min exposure | **off** | off |
| 4 | For every substance (e.g. H₂O, then H₂O₂): bus on, optional valve, pump 60 s, bus off, wait 60 s, 30 min exposure | on → **off** | off |

Then the next position, and optionally the next cycle over all positions.

**How the notebook runs it.** The experiment runs in a background thread. Interrupting or
re-running the *dashboard* cell does not stop it, and after a browser reconnect you just re-run the
dashboard. Stop it only with the **Stop** cell. Every acquisition is logged to disk
(`results.jsonl`, `temperature.csv`, preview PNGs), so the log survives a lost browser tab. The
full-bit-depth TIFFs are written by ImSwitch to its data folder under `recordings/<date>/`.

**Before you start**

- Cutting bus power switches off everything on the CAN bus: motor drivers, LED/laser boards, the
  pump and the valve. The stage is unpowered during each exposure. After the bus comes back on, the
  notebook re-reads the stage position and by default **aborts** if it moved or the counter reset.
- Positions are **absolute** X/Y coordinates in µm (the user frame of `getPositionerPositions`).
  Check them on the live view first, e.g. with the "record current position" cell.
- Do a `DRY_RUN = True` pass first. It uses short exposures and waits and runs through every step
  in a few minutes.

Needs `imswitchclient` with `recordingManager.snapAndWait`, `uc2ConfigManager` and
`liveViewManager` (openUC2/imswitchclient after 0.1.7).

## 1 Connect

In [ ]:
import os, json, time, threading, traceback, datetime, csv
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

import imswitchclient.ImSwitchClient as imc

# None = $IMSWITCH_API_URL, which ImSwitch sets for the notebooks it serves itself (that is
# the internal port; port 80 is only the outside view through Caddy).
# From another computer use the address you open in the browser, e.g.
#   IMSWITCH_URL = "http://100.69.203.113/imswitch/api"
IMSWITCH_URL = None

client = imc.ImSwitchClient(IMSWITCH_URL)
print("Connected to:", client.base_uri)

for need in ("uc2ConfigManager", "liveViewManager"):
    if not hasattr(client, need):
        raise ImportError(f"imswitchclient is too old (no {need}) - update it, e.g. "
                          "pip install -U https://github.com/openUC2/imswitchclient/archive/refs/heads/main.zip")

DETECTOR = client.settingsManager.getDetectorNames()[0]
STAGE = client.positionersManager.allStageNames[0]
print("Detector:", DETECTOR, "| Stage:", STAGE)
print("Laser/PWM channels:", {n: client.lasersManager.getLaserValueRanges(n)
                              for n in client.lasersManager.getLaserNames()})
print("Bus:", client.uc2ConfigManager.getBusStatus())
print("Stage position:", client.positionersManager.getPositionerPositions()[STAGE])

## 2 Configuration

Channel names are the laser names from the setup JSON, as printed above (e.g. `"LASER1"` for PWM
channel 1). Values are in that channel's range, 0..1023 for a UC2 PWM output.

In [ ]:
RUN_NAME = "darkcycle"
DRY_RUN = True             # True: short exposures/waits to test the sequence end to end

# --- camera, dark acquisitions ---------------------------------------------------------
DARK_EXPOSURE_MS = 30 * 60 * 1000      # 30 min
DARK_GAIN = 23                         # Toupcam UI scale 0..23 (23 = highest)
TARGET_TEMPERATURE_C = -30.0
COOLDOWN_TOLERANCE_C = 1.0             # start once |T - target| is below this ...
COOLDOWN_TIMEOUT_S = 30 * 60           # ... or after this long (then continue with a warning)
CAMERA_EXTRA = {}                      # further parameters, e.g. {"blacklevel": 10, "conversion_gain": "HCG"}

# --- brightfield reference -------------------------------------------------------------
LED = {"type": "laser", "name": "LED", "value": 1023}
# LED-matrix instead:  LED = {"type": "ledmatrix", "intensity": 255}
BF_GAIN = 0
BF_START_EXPOSURE_MS = 20
BF_AUTO_EXPOSURE = True                # camera auto exposure on the live stream, then frozen
BF_AE_SETTLE_S = 5

# --- fluidics (laser/PWM channels) -------------------------------------------------------
# One entry per substance, run in this order after the dark baseline.
#   pump  - channel that drives the pump, run at pump_value for pump_s seconds
#   valve - optional channel switched on (valve_value) while this substance is pumped
SUBSTANCES = [
    {"name": "H2O",  "pump": "LASER1", "pump_value": 1023, "pump_s": 60, "valve": None},
    {"name": "H2O2", "pump": "LASER2", "pump_value": 1023, "pump_s": 60,
     "valve": "LASER0", "valve_value": 1023},
]

# --- bus power ---------------------------------------------------------------------------
BUS_OFF_WAIT_S = 60          # bus off -> wait -> expose
BUS_ON_SETTLE_S = 5          # bus on  -> wait for the CAN slaves to boot before using them
REQUIRE_BUS_POWER = True     # False: continue if setBusPower fails (virtual microscope)

# --- positions (absolute, µm, user frame) -------------------------------------------------
POSITIONS = [
    {"name": "fov1", "x": 0.0,    "y": 0.0},
    {"name": "fov2", "x": 1000.0, "y": 0.0},
]
MOVE_SPEED = 5000
POSITION_TOLERANCE_UM = 50   # allowed X/Y change across a bus power cycle
ON_POSITION_MISMATCH = "abort"   # "abort" or "warn"
N_CYCLES = 1                 # passes over all positions

# --- logging / display -----------------------------------------------------------------
TEMP_POLL_S = 30
SNAP_POLL_S = 5
FULL_IMAGE_STATS = True      # download each dark TIFF to log mean/std/max (16 bit)
OUTPUT_ROOT = Path.home() / "longterm_dark_runs"

if DRY_RUN:
    DARK_EXPOSURE_MS = 3000
    BUS_OFF_WAIT_S, BUS_ON_SETTLE_S = 3, 2
    COOLDOWN_TIMEOUT_S = 20
    REQUIRE_BUS_POWER = False
    SUBSTANCES = [{**s, "pump_s": min(s["pump_s"], 3)} for s in SUBSTANCES]
    SNAP_POLL_S = 1
    TEMP_POLL_S = 5

# --- sanity checks ------------------------------------------------------------------------
_channels = set(client.lasersManager.getLaserNames())
_needed = {s["pump"] for s in SUBSTANCES} | {s["valve"] for s in SUBSTANCES if s.get("valve")}
if LED["type"] == "laser":
    _needed.add(LED["name"])
_missing = _needed - _channels
if _missing:
    raise ValueError(f"Unknown laser channels {_missing}; available: {sorted(_channels)}")

_per_position_s = (BF_AE_SETTLE_S + 10
                   + (1 + len(SUBSTANCES)) * (BUS_OFF_WAIT_S + DARK_EXPOSURE_MS / 1000 + 15)
                   + sum(s["pump_s"] + BUS_ON_SETTLE_S for s in SUBSTANCES))
_total_s = _per_position_s * len(POSITIONS) * N_CYCLES
print(f"{'DRY RUN - ' if DRY_RUN else ''}{len(POSITIONS)} positions x {N_CYCLES} cycle(s), "
      f"{1 + len(SUBSTANCES)} dark frames per position")
print(f"Estimated duration: {_per_position_s / 3600:.2f} h per position, {_total_s / 3600:.1f} h total")
for p in POSITIONS:
    print(f"  {p['name']}: X={p['x']:.1f} µm  Y={p['y']:.1f} µm")

### Optional: record the current stage position

Move to a field of view in the ImSwitch UI, then run this cell to append it to `POSITIONS`.

In [ ]:
_pos = client.positionersManager.getPositionerPositions()[STAGE]
POSITIONS.append({"name": f"fov{len(POSITIONS) + 1}", "x": float(_pos["X"]), "y": float(_pos["Y"])})
POSITIONS

## 3 Camera: apply dark settings and start cooling

Running this cell again applies changed values. `CAMERA_EXTRA` is applied through the typed
parameter setter, so any name from `getDetectorParameterTree` works there.

In [ ]:
def apply_dark_settings():
    s = client.settingsManager
    s.setDetectorMode(DETECTOR, False)
    s.setDetectorGain(DETECTOR, DARK_GAIN)
    s.setDetectorExposureTime(DETECTOR, DARK_EXPOSURE_MS)   # also stops the live stream
    for name, value in CAMERA_EXTRA.items():
        s.setDetectorParameterValue(name, value, detector_name=DETECTOR)

apply_dark_settings()
client.settingsManager.setCameraTargetTemperature(TARGET_TEMPERATURE_C, detector_name=DETECTOR)

_readback = {k: client.settingsManager.getDetectorParameter(k, DETECTOR)
             for k in ("exposure", "gain", "exposure_mode")}
print("Camera read back:", _readback)
if _readback["exposure"] is not None and abs(_readback["exposure"] - DARK_EXPOSURE_MS) > 1:
    print(f"WARNING: camera runs at {_readback['exposure']} ms, not {DARK_EXPOSURE_MS} ms "
          "(clamped to the sensor's maximum exposure?)")
print("Temperature:", client.settingsManager.getCameraTemperature(DETECTOR))

## 4 Temperature logger

Polls the sensor temperature in a background thread and appends it to `temperature.csv` in the
run folder. It keeps running between experiments; `temp_logger.stop()` ends it.

In [ ]:
RUN_DIR = OUTPUT_ROOT / f"{datetime.datetime.now():%Y-%m-%d_%H-%M-%S}_{RUN_NAME}"
(RUN_DIR / "previews").mkdir(parents=True, exist_ok=True)
print("Run folder:", RUN_DIR)


class TemperatureLogger:
    def __init__(self, path, period_s):
        self.path, self.period_s = Path(path), period_s
        self.samples = []            # (unix time, temperature °C, target °C)
        self._stop = threading.Event()
        self._thread = threading.Thread(target=self._loop, daemon=True, name="TempLogger")

    def read(self):
        t = client.settingsManager.getCameraTemperature(DETECTOR)
        return time.time(), t.get("temperature"), t.get("target")

    def latest(self):
        return self.samples[-1] if self.samples else (None, None, None)

    def _loop(self):
        new = not self.path.exists()
        with open(self.path, "a", newline="") as f:
            w = csv.writer(f)
            if new:
                w.writerow(["iso_time", "unix_time", "temperature_c", "target_c"])
            while not self._stop.is_set():
                try:
                    ts, temp, target = self.read()
                    self.samples.append((ts, temp, target))
                    w.writerow([datetime.datetime.fromtimestamp(ts).isoformat(), ts, temp, target])
                    f.flush()
                except Exception as e:      # a missed sample must not end the log
                    print("temperature read failed:", e)
                self._stop.wait(self.period_s)

    def start(self):
        self._thread.start()
        return self

    def stop(self):
        self._stop.set()


def plot_temperature(ax, samples):
    ax.clear()
    pts = [(ts, temp, target) for ts, temp, target in samples if temp is not None]
    if not pts:
        ax.text(0.5, 0.5, "no temperature reported", ha="center", va="center", transform=ax.transAxes)
        return
    t0 = pts[0][0]
    tt = [(ts - t0) / 60 for ts, _, _ in pts]
    ax.plot(tt, [p[1] for p in pts], label="sensor")
    if any(p[2] is not None for p in pts):
        ax.plot(tt, [p[2] for p in pts], "--", label="target")
    ax.set_xlabel("time [min]")
    ax.set_ylabel("°C")
    ax.set_title(f"Sensor temperature: {pts[-1][1]:.1f} °C")
    ax.legend(loc="upper right")
    ax.grid(alpha=0.3)


try:
    temp_logger.stop()      # re-running the cell replaces the old logger
except NameError:
    pass
temp_logger = TemperatureLogger(RUN_DIR / "temperature.csv", TEMP_POLL_S).start()
time.sleep(1)
print("First sample:", temp_logger.latest())

## 5 Experiment runner

Defines the sequence. Nothing moves until you start it in the next section.

In [ ]:
class DarkCycleRunner:
    def __init__(self):
        self.stop_event = threading.Event()
        self.state = {"phase": "idle", "detail": "", "cycle": None, "position": None,
                      "progress": None, "error": None, "started": None, "finished": None}
        self.results = []          # one dict per acquisition, same as results.jsonl
        self.events = []
        self._thread = None
        self._last_xy = None

    # --- bookkeeping ---------------------------------------------------------------
    def log(self, msg):
        line = f"{datetime.datetime.now():%H:%M:%S}  {msg}"
        self.events.append(line)
        with open(RUN_DIR / "events.log", "a") as f:
            f.write(line + "\n")

    def phase(self, phase, detail=""):
        self.state.update(phase=phase, detail=detail, progress=None)
        self.log(f"[{phase}] {detail}")

    def check_stop(self):
        if self.stop_event.is_set():
            raise InterruptedError("stopped by user")

    def wait(self, seconds, what):
        t_end = time.time() + seconds
        while (remaining := t_end - time.time()) > 0:
            self.check_stop()
            self.state["detail"] = f"{what}: {remaining:.0f} s left"
            self.stop_event.wait(min(1.0, remaining))
        self.check_stop()

    # --- hardware steps --------------------------------------------------------------
    def bus(self, enable):
        try:
            client.uc2ConfigManager.setBusPower(enable)
        except Exception as e:
            if REQUIRE_BUS_POWER:
                raise
            self.log(f"bus power {'on' if enable else 'off'} failed, continuing: {e}")

    def bus_on(self):
        self.phase("bus on", f"settling {BUS_ON_SETTLE_S} s")
        self.bus(True)
        self.wait(BUS_ON_SETTLE_S, "bus settling")
        self.verify_position()

    def bus_off_and_wait(self):
        self.phase("bus off", f"waiting {BUS_OFF_WAIT_S} s")
        self.bus(False)
        self.wait(BUS_OFF_WAIT_S, "bus off, waiting")

    def verify_position(self):
        if self._last_xy is None:
            return
        pos = client.positionersManager.getPositionerPositions()[STAGE]
        dx, dy = pos["X"] - self._last_xy[0], pos["Y"] - self._last_xy[1]
        if max(abs(dx), abs(dy)) > POSITION_TOLERANCE_UM:
            msg = (f"stage reads X={pos['X']:.1f} Y={pos['Y']:.1f} after the bus power cycle, "
                   f"expected X={self._last_xy[0]:.1f} Y={self._last_xy[1]:.1f}")
            if ON_POSITION_MISMATCH == "abort":
                raise RuntimeError(msg)
            self.log("WARNING " + msg)

    def move_to(self, p):
        self.phase("move", f"{p['name']} -> X={p['x']:.1f} Y={p['y']:.1f} µm")
        pm = client.positionersManager
        for axis in ("X", "Y"):
            pm.movePositioner(STAGE, axis, p[axis.lower()], is_absolute=True, is_blocking=True,
                              speed=MOVE_SPEED)
        pos = pm.getPositionerPositions()[STAGE]
        self._last_xy = (pos["X"], pos["Y"])
        self.log(f"at X={pos['X']:.1f} Y={pos['Y']:.1f} Z={pos.get('Z', float('nan')):.1f}")

    def led(self, on):
        if LED["type"] == "ledmatrix":
            if on:
                client.ledMatrixManager.setAllLED(state=1, intensity=LED["intensity"])
            else:
                client.ledMatrixManager.setAllLEDOff()
        elif on:
            client.lasersManager.setLaserOn(LED["name"], LED["value"])
        else:
            client.lasersManager.setLaserOff(LED["name"])

    def all_outputs_off(self):
        names = {s["pump"] for s in SUBSTANCES} | {s["valve"] for s in SUBSTANCES if s.get("valve")}
        for name in names:
            client.lasersManager.setLaserOff(name)
        self.led(False)

    def snap(self, label, cycle, p, kind):
        name = f"{RUN_NAME}_c{cycle:03d}_{p['name']}_{label}"
        t_before = temp_logger.read()[1]
        t0 = time.time()

        def progress(status):
            self.state["progress"] = status.get("progress")
            elapsed = status.get("elapsedMs", 0) / 1000
            self.state["detail"] = (f"{label}: {status.get('status')} {elapsed / 60:.1f} / "
                                    f"{status.get('expectedDurationMs', 0) / 60000:.1f} min")

        result = client.recordingManager.snapAndWait(name, poll=SNAP_POLL_S, on_progress=progress,
                                                     stop_event=self.stop_event)
        entry = {"cycle": cycle, "position": p["name"], "x": p["x"], "y": p["y"], "label": label,
                 "kind": kind, "start": datetime.datetime.fromtimestamp(t0).isoformat(),
                 "duration_s": round(time.time() - t0, 1),
                 "exposure_ms": client.settingsManager.getDetectorParameter("exposure", DETECTOR),
                 "gain": client.settingsManager.getDetectorParameter("gain", DETECTOR),
                 "temperature_start_c": t_before, "temperature_end_c": temp_logger.read()[1],
                 "fullPath": result.get("fullPath"), "relativeFilePath": result.get("relativeFilePath")}
        preview = client.recordingManager.previewToNumpy(result)
        if preview is not None:
            entry["preview_png"] = str(RUN_DIR / "previews" / f"{name}.png")
            plt.imsave(entry["preview_png"], preview, cmap="gray")
        if kind == "dark" and FULL_IMAGE_STATS and result.get("relativeFilePath"):
            try:
                img = client.recordingManager.downloadImage(result["relativeFilePath"]).astype(np.float64)
                entry.update(mean=float(img.mean()), std=float(img.std()), max=float(img.max()))
            except Exception as e:
                self.log(f"could not read {result['relativeFilePath']} for statistics: {e}")
        with open(RUN_DIR / "results.jsonl", "a") as f:
            f.write(json.dumps(entry) + "\n")
        entry["preview"] = preview
        self.results.append(entry)
        self.log(f"saved {label}: {entry['fullPath']}")
        return entry

    def brightfield(self, cycle, p):
        self.phase("brightfield", p["name"])
        s, lv = client.settingsManager, client.liveViewManager
        s.setDetectorGain(DETECTOR, BF_GAIN)
        s.setDetectorExposureTime(DETECTOR, BF_START_EXPOSURE_MS)
        self.led(True)
        try:
            if BF_AUTO_EXPOSURE:
                s.setDetectorMode(DETECTOR, True)
                lv.startLiveView(DETECTOR)
                self.wait(BF_AE_SETTLE_S, "auto exposure")
                s.setDetectorMode(DETECTOR, False)       # freezes the exposure AE settled on
                lv.stopLiveView(DETECTOR)
                exposure = s.getDetectorParameter("exposure", DETECTOR)
                if exposure:
                    # keep ImSwitch's cached value in sync so the snap waits long enough
                    s.setDetectorExposureTime(DETECTOR, exposure)
                self.log(f"auto exposure settled at {exposure} ms")
            self.snap("brightfield", cycle, p, kind="brightfield")
        finally:
            self.led(False)
            s.setDetectorMode(DETECTOR, False)
            try:
                lv.stopLiveView(DETECTOR)
            except Exception:
                pass
            apply_dark_settings()

    def dark(self, label, cycle, p):
        self.bus_off_and_wait()
        self.phase("dark exposure", f"{p['name']} / {label}")
        self.snap(label, cycle, p, kind="dark")

    def fluidics(self, sub):
        self.phase("pump", f"{sub['name']}: {sub['pump']} @ {sub['pump_value']} for {sub['pump_s']} s")
        valve = sub.get("valve")
        if valve:
            client.lasersManager.setLaserOn(valve, sub.get("valve_value", 1023))
        try:
            def tick(elapsed):
                self.state["detail"] = f"pumping {sub['name']}: {elapsed:.0f} / {sub['pump_s']} s"
            client.lasersManager.runLaserFor(sub["pump"], sub["pump_value"], sub["pump_s"],
                                             on_tick=tick, stop_event=self.stop_event)
        finally:
            if valve:
                client.lasersManager.setLaserOff(valve)
        self.check_stop()

    def cooldown(self):
        self.phase("cooldown", f"target {TARGET_TEMPERATURE_C} °C")
        t_end = time.time() + COOLDOWN_TIMEOUT_S
        while time.time() < t_end:
            self.check_stop()
            temp = temp_logger.read()[1]
            if temp is None:
                self.log("camera reports no temperature, skipping cooldown")
                return
            self.state["detail"] = f"sensor {temp:.1f} °C, target {TARGET_TEMPERATURE_C} °C"
            if abs(temp - TARGET_TEMPERATURE_C) <= COOLDOWN_TOLERANCE_C:
                self.log(f"sensor at {temp:.1f} °C")
                return
            self.stop_event.wait(min(TEMP_POLL_S, 10))
        self.log(f"WARNING cooldown timed out at {temp_logger.read()[1]} °C, continuing")

    # --- sequence -----------------------------------------------------------------------
    def run(self):
        self.state.update(started=datetime.datetime.now().isoformat(), error=None)
        try:
            self.bus(True)
            apply_dark_settings()
            self.cooldown()
            for cycle in range(N_CYCLES):
                for p in POSITIONS:
                    self.state.update(cycle=cycle, position=p["name"])
                    self.bus_on()
                    self.move_to(p)
                    self.brightfield(cycle, p)
                    self.dark("dark_baseline", cycle, p)
                    for sub in SUBSTANCES:
                        self.bus_on()
                        self.fluidics(sub)
                        self.dark(f"dark_{sub['name']}", cycle, p)
            self.phase("done", f"{len(self.results)} acquisitions")
        except InterruptedError:
            self.phase("stopped", "stopped by user")
        except Exception as e:
            self.state["error"] = f"{e}"
            self.phase("error", f"{e}")
            self.log(traceback.format_exc())
        finally:
            self.state["finished"] = datetime.datetime.now().isoformat()
            # leave the instrument powered, dark and with every actuator off
            try:
                self.bus(True)
                time.sleep(BUS_ON_SETTLE_S)
                self.all_outputs_off()
            except Exception as e:
                self.log(f"cleanup failed: {e}")

    def start(self):
        if self.running:
            raise RuntimeError("already running")
        self.stop_event.clear()
        self._thread = threading.Thread(target=self.run, daemon=True, name="DarkCycle")
        self._thread.start()
        return self

    def stop(self):
        """Cancel the running exposure / pump and end after cleanup."""
        self.stop_event.set()

    @property
    def running(self):
        return self._thread is not None and self._thread.is_alive()


with open(RUN_DIR / "config.json", "w") as f:
    json.dump({k: v for k, v in globals().items()
               if k.isupper() and isinstance(v, (int, float, str, bool, list, dict, type(None)))},
              f, indent=2, default=str)
print("Config saved to", RUN_DIR / "config.json")

## 6 Start

Starts the run in the background and returns immediately.

In [ ]:
try:
    if runner.running:
        raise RuntimeError("A run is still active - stop it first (section 8)")
except NameError:
    pass
runner = DarkCycleRunner().start()
print("Started. Run folder:", RUN_DIR)

## 7 Dashboard

Refreshes every few seconds: the current step, the temperature trace and the latest images.
Interrupting this cell (■) only stops the display, not the experiment. Re-run it at any time,
for example after the browser reconnects.

In [ ]:
DASH_REFRESH_S = 10
N_IMAGES = 4         # latest acquisitions shown


def show_dashboard():
    st = runner.state
    fig = plt.figure(figsize=(14, 7))
    gs = fig.add_gridspec(2, N_IMAGES)
    plot_temperature(fig.add_subplot(gs[0, :]), temp_logger.samples)
    for i, entry in enumerate(runner.results[-N_IMAGES:]):
        ax = fig.add_subplot(gs[1, i])
        if entry.get("preview") is not None:
            ax.imshow(entry["preview"], cmap="gray")
        title = f"{entry['position']} {entry['label']}"
        if "mean" in entry:
            title += f"\nmean {entry['mean']:.1f} std {entry['std']:.1f}"
        ax.set_title(title, fontsize=9)
        ax.axis("off")
    prog = "" if st["progress"] is None else f" ({st['progress']:.0f}%)"
    fig.suptitle(f"cycle {st['cycle']} | {st['position']} | {st['phase']}{prog} | {st['detail']}",
                 fontsize=11)
    fig.tight_layout()
    plt.show()
    print("\n".join(runner.events[-8:]))
    if st["error"]:
        print("ERROR:", st["error"])


try:
    while True:
        clear_output(wait=True)
        show_dashboard()
        if not runner.running:
            print("Runner finished:", runner.state["phase"])
            break
        time.sleep(DASH_REFRESH_S)
except KeyboardInterrupt:
    print("Dashboard stopped - the experiment keeps running.")

## 8 Stop

Interrupt the dashboard cell first (■), because the kernel runs one cell at a time. This cell
cancels the exposure or pump that is running, powers the bus back on and switches every output
off. Files already written stay on disk.

In [ ]:
runner.stop()
runner._thread.join(timeout=120)
print(runner.state["phase"], runner.state["detail"])

## 9 Review

Reads the logs back from disk, so it works for any earlier run folder too. Set `REVIEW_DIR` to
look at another run.

In [ ]:
REVIEW_DIR = RUN_DIR

entries = [json.loads(l) for l in open(REVIEW_DIR / "results.jsonl")] \
    if (REVIEW_DIR / "results.jsonl").exists() else []
print(f"{len(entries)} acquisitions in {REVIEW_DIR}")

temps = []
if (REVIEW_DIR / "temperature.csv").exists():
    with open(REVIEW_DIR / "temperature.csv") as f:
        for row in csv.DictReader(f):
            temp = float(row["temperature_c"]) if row["temperature_c"] not in ("", "None") else None
            target = float(row["target_c"]) if row["target_c"] not in ("", "None") else None
            temps.append((float(row["unix_time"]), temp, target))
fig, ax = plt.subplots(figsize=(12, 3))
plot_temperature(ax, temps)
plt.show()

positions = list(dict.fromkeys((e["cycle"], e["position"]) for e in entries))
labels = list(dict.fromkeys(e["label"] for e in entries))
if entries:
    fig, axes = plt.subplots(len(positions), len(labels),
                             figsize=(3.2 * len(labels), 3.2 * len(positions)), squeeze=False)
    for r, key in enumerate(positions):
        for c, label in enumerate(labels):
            ax = axes[r, c]
            ax.axis("off")
            e = next((e for e in entries if (e["cycle"], e["position"]) == key and e["label"] == label), None)
            if e is None:
                continue
            if e.get("preview_png") and Path(e["preview_png"]).exists():
                ax.imshow(plt.imread(e["preview_png"]), cmap="gray")
            t = f"c{key[0]} {key[1]}\n{label}"
            if "mean" in e:
                t += f"\nmean {e['mean']:.1f} std {e['std']:.1f}"
            ax.set_title(t, fontsize=8)
    fig.tight_layout()
    plt.show()

for e in entries:
    print(f"c{e['cycle']} {e['position']:>6} {e['label']:<16} {e.get('mean', float('nan')):9.2f} "
          f"{e.get('temperature_start_c')} °C  {e['fullPath']}")